In [ ]:
from helperfunctions.intern_constants import TRAIN_START, TRAIN_END, PATH_PC_FILTERING
import pandas as pd
from glob import glob
import os
import pandas as pd

In [2]:
n_train_theory = len(pd.date_range(TRAIN_START,TRAIN_END,freq="10min"))

n_list = []
all_wts = []
files = glob(os.path.join(PATH_PC_FILTERING, "*.csv"))

for fp in files:
    df = pd.read_csv(fp)
    n_list.append(len(df))
    all_wts.append(df)
print(f"available timestamps per wt after power curve filtering:\n{n_list}")

df_all = pd.concat(all_wts,ignore_index=True)

available timestamps per wt after power curve filtering:
[36714, 37852, 35732, 39616, 38998, 36532, 39018, 38523, 38024, 37671, 37695, 36643, 36195, 35754]


In [3]:
df_all = df_all.sort_index()
df_all.index.is_unique
print(len(df_all))

524967


In [4]:
df_all.index

RangeIndex(start=0, stop=524967, step=1)

In [5]:
df_all.head()

,Date and time,WT_ID,Ambient temperature (converter) (°C),Drive train acceleration (mm/ss),Gear oil inlet pressure (bar),Gear oil pump pressure (bar),Gearbox speed (RPM),Generator bearing front temperature (°C),Generator bearing rear temperature (°C),Generator RPM (RPM),...,Blade angle (pitch position) C (°),Front bearing temperature (°C),Gear oil inlet temperature (°C),Gear oil temperature (°C),Rear bearing temperature (°C),Tower Acceleration X (mm/ss),Tower Acceleration Y (mm/ss),Transformer cell temperature (°C),Transformer temperature (°C),Yaw bearing angle (°)
0,2018-04-05 13:50:00,10,9.325000,83.771255,73.893341,278.424957,1367.796143,38.070000,38.689999,1367.280396,...,0.000000,63.134998,41.215000,53.450001,58.599998,95.407021,24.838570,11.405000,37.424999,263.073761
1,2018-04-05 14:00:00,10,9.421667,74.655060,73.764961,277.703827,1315.326660,38.788334,39.311665,1314.831421,...,0.280657,63.448334,41.511665,53.576668,58.849998,80.319031,24.520512,11.480000,37.391666,255.862030
2,2018-04-05 14:10:00,10,9.678333,80.697540,72.869965,275.970703,1491.923584,39.888332,40.183334,1491.143188,...,0.131996,62.490002,41.985001,53.533333,58.256668,70.224258,26.170935,11.746667,37.276665,253.195679
3,2018-04-05 14:20:00,10,9.548333,89.827606,66.711533,253.797516,1482.711548,39.688332,40.148335,1482.276367,...,0.000000,66.991669,45.573334,54.783333,62.139999,70.493835,23.590015,11.220000,37.953335,253.195679
4,2018-04-05 14:30:00,10,9.566667,89.401253,65.972282,257.082275,1405.899658,38.044998,38.619999,1405.585571,...,0.049332,66.074997,47.993332,54.881668,61.341667,72.243156,27.721466,11.208333,38.056667,253.195679


In [6]:
df_all["Date and time"] = pd.to_datetime(df_all["Date and time"], errors="raise")
df_all = df_all.set_index("Date and time")
df_all = df_all.sort_index()
df_all.head()
df_all.index

DatetimeIndex(['2018-04-05 13:50:00', '2018-04-05 13:50:00',
               '2018-04-05 13:50:00', '2018-04-05 13:50:00',
               '2018-04-05 13:50:00', '2018-04-05 13:50:00',
               '2018-04-05 13:50:00', '2018-04-05 13:50:00',
               '2018-04-05 13:50:00', '2018-04-05 13:50:00',
               ...
               '2019-04-05 13:50:00', '2019-04-05 13:50:00',
               '2019-04-05 13:50:00', '2019-04-05 13:50:00',
               '2019-04-05 13:50:00', '2019-04-05 13:50:00',
               '2019-04-05 13:50:00', '2019-04-05 13:50:00',
               '2019-04-05 13:50:00', '2019-04-05 13:50:00'],
              dtype='datetime64[ns]', name='Date and time', length=524967, freq=None)

In [7]:
idx = pd.DatetimeIndex(df_all.index.unique()).sort_values()
diffs = idx.to_series().diff().dropna()
gaps = diffs[diffs > pd.Timedelta("10min")]

has_gaps = not gaps.empty

n_missing_points = int(((gaps / pd.Timedelta("10min"))-1).round().astype(int).sum())

print(has_gaps, n_missing_points)

True 6989


In [8]:
missing_ts = []
prev = idx[:-1]
curr = idx[1:]

mask = (curr - prev) > pd.Timedelta("10min")

for p, c in zip(prev[mask], curr[mask]):
    rng = pd.date_range(
        p + pd.Timedelta("10min"),
        c - pd.Timedelta("10min"),
        freq=pd.Timedelta("10min")
    )
    missing_ts.extend(rng)
    
missing_ts = pd.DatetimeIndex(missing_ts)

print(missing_ts)

DatetimeIndex(['2018-04-05 19:00:00', '2018-04-05 19:10:00',
               '2018-04-07 11:00:00', '2018-04-07 11:10:00',
               '2018-04-07 11:20:00', '2018-04-07 11:30:00',
               '2018-04-07 11:40:00', '2018-04-07 11:50:00',
               '2018-04-07 12:10:00', '2018-04-07 12:50:00',
               ...
               '2019-04-02 19:00:00', '2019-04-03 11:00:00',
               '2019-04-03 11:30:00', '2019-04-03 11:50:00',
               '2019-04-03 12:00:00', '2019-04-03 12:10:00',
               '2019-04-03 12:20:00', '2019-04-03 12:30:00',
               '2019-04-03 12:40:00', '2019-04-03 12:50:00'],
              dtype='datetime64[ns]', length=6989, freq=None)
